# ETH Rapid Reversal — Signal Walkthrough

Step-by-step demonstration of the signal logic on a single day of ETH 5m data.

Sections:
1. Load data (CMC or synthetic)
2. Compute indicators (RSI, MACD, ADX, EMA)
3. Inspect a sample signal
4. Visualize the trade

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt

from backtest.indicators import rsi, macd, adx, ema, bollinger_bands, volume_above_average, fear_greed_alignment, fear_greed_zone
from backtest.data_loader import load_cmc_ohlcv, load_cmc_fear_greed

In [ ]:
df = load_cmc_ohlcv('ETH', '5m', days=2)
fg = load_cmc_fear_greed(days=2)
df['rsi'] = rsi(df['close'], 14)
df[['macd', 'signal', 'hist']] = macd(df['close'])
df['ema50'] = ema(df['close'], 50)
df['bb_upper'], df['bb_mid'], df['bb_lower'] = bollinger_bands(df['close'], 20, 2.0)
df['vol_ok'] = volume_above_average(df['volume'], 20, 1.0).astype(float)
df.tail()

In [ ]:
fg_value = float(fg.iloc[-1])
print(f'Current Fear & Greed: {fg_value:.1f}  ({fear_greed_zone(fg_value)})')
print(f'Long alignment:  {fear_greed_alignment(fg_value, "long")}')
print(f'Short alignment: {fear_greed_alignment(fg_value, "short")}')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
axes[0].plot(df['timestamp'], df['close'], label='ETH 5m close', color='#1f77b4')
axes[0].plot(df['timestamp'], df['ema50'], label='EMA50', color='#ff7f0e', alpha=0.6)
axes[0].plot(df['timestamp'], df['bb_upper'], label='BB upper', color='#9467bd', alpha=0.4)
axes[0].plot(df['timestamp'], df['bb_lower'], label='BB lower', color='#9467bd', alpha=0.4)
axes[0].set_ylabel('Price')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(df['timestamp'], df['rsi'], label='RSI(14)', color='#2ca02c')
axes[1].axhline(35, color='gray', linestyle='--', alpha=0.5)
axes[1].axhline(65, color='gray', linestyle='--', alpha=0.5)
axes[1].set_ylabel('RSI'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].bar(df['timestamp'], df['hist'], label='MACD hist', color='#d62728', alpha=0.5)
axes[2].set_ylabel('MACD hist'); axes[2].legend(); axes[2].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Notes

- Long signal: `rsi <= 35` AND `hist rising 2 bars` AND `price within 1.5% of lower BB` AND `volume > 20-bar avg` AND `price > EMA50` AND `ADX >= 15` AND `F&G not hostile` AND `funding not extreme`
- Short signal: mirror conditions (`rsi >= 65`, upper BB, price < EMA50)
- Tier A (high conviction): `rsi <= 22` (or `>= 78` for short) + MACD cross + F&G tailwind
- Exit: trailing stop (activates at +0.5% ROE, trails 0.3% behind high water mark); hard SL at -1.0% ROE; immediate exit when ADX < 15; MACD reversal (3 bars against, after momentum confirmed)

See `SKILL.md` for the full rule set, `backtest/run_backtest.py` for the full replay, and `demo.py` for a one-command summary.